In [16]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, median_absolute_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone
from scipy.stats import randint, uniform
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import glob, os
import time
import sys
sys.path.append('..')
from src.preprocessing import FeatureExtractor
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
mlflow.set_experiment("kufar-notebook")

def log_model(model, X_train, y_train, X_test, y_test, run_name="run", log=False):
    
    if hasattr(model, "best_estimator_"):
        params = model.best_params_
        model = model.best_estimator_
    else:
        params = model.get_params()

    train_pred = model.predict(X_train)
    
    if log:
        y_pred = np.exp(model.predict(X_test))
    else:
        y_pred = model.predict(X_test)
        
    errors = y_test - y_pred

    metrics = {
        "train_mae": mean_absolute_error(y_train, train_pred),
        "train_r2": r2_score(y_train, train_pred),
        "test_mae": mean_absolute_error(y_test, y_pred),
        "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "test_r2": r2_score(y_test, y_pred),
        "test_mape": mean_absolute_percentage_error(y_test, y_pred),
        "test_median_ae": median_absolute_error(y_test, y_pred),
        "error_p95": np.percentile(np.abs(errors), 95),
    }

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, "model")

### Чтение данных

In [3]:
folder_path = r'..\data\raw'
file_type = '/*csv'

files = glob.glob(folder_path + file_type)

latest_file = max(files, key=os.path.getctime)

time = latest_file[12:].split(sep="_")

access_time = pd.Timestamp(
    year=int(time[0][0:4]),
    month=int(time[0][4:6]),
    day=int(time[0][6:8]),
    hour=int(time[1][0:2]),
    minute=int(time[1][2:4]),
    tz='Europe/Moscow'
)

df = pd.read_csv(latest_file)
df =  df[df["price_byn"] > 30]

### Разбиение на тестовую и обучающую выборки

In [4]:
y = df["price_byn"]
X = df.drop(["price_byn"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

### Функция обучения и оценки пайплайна

In [5]:
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

# таблица с результатами
results = []


def evaluate_model(name, pipeline, X_train, y_train, X_test, y_test, log=False):

    if log:
        pipeline.fit(X_train, np.log(y_train))
        y_pred = np.exp(pipeline.predict(X_test))
    else:
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    cv = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring={
            "r2": "r2",
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error"
        },
        n_jobs=-1,
        return_train_score=False
    )

    results.append({
        "Model": name,

        "Test R2": r2,
        "Test RMSE": rmse,
        "Test MAE": mae,
        "Test MAPE": mape,

        "CV R2": cv["test_r2"].mean(),
        "CV RMSE": -cv["test_rmse"].mean(),
        "CV MAE": -cv["test_mae"].mean(),
    })


def show_results():

    df = pd.DataFrame(results)

    if df.empty:
        print("Нет результатов")
        return

    numeric_cols = df.select_dtypes(include=np.number).columns

    return (
        df.style
        .format("{:.4f}", subset=numeric_cols)

        # где больше — лучше
        .highlight_max(
            subset=["Test R2", "CV R2"],
            color="lightgreen"
        )

        # где меньше — лучше
        .highlight_min(
            subset=[
                "Test RMSE",
                "Test MAE",
                "Test MAPE",
                "CV RMSE",
                "CV MAE"
            ],
            color="lightgreen"
        )
    )

In [12]:
categorical_features = [
    "brand",
    "processor",
    "rom_type",
    "os", 
    "videocard",
    "videocard_brand",
    "region",
    "matrix_type",
    "display_resolution",
    "ram_type"
]

cb_pipeline = Pipeline([
    ('extractor', FeatureExtractor(access_time)),
    ('regressor', CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=7,
        loss_function='MAE',
        early_stopping_rounds=50,
        random_seed=42,
        logging_level='Silent',
        cat_features=categorical_features
    ))
])

xgb_pipeline = Pipeline([
    ('extractor', FeatureExtractor(access_time, GBM_XGB=True)),
    ('regressor', XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=7,
        objective='reg:absoluteerror',
        random_state=42,
        enable_categorical=True
    ))
])


lgb_pipeline = Pipeline([
    ('extractor', FeatureExtractor(access_time, GBM_XGB=True)),
    ('regressor', LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=7,
        objective='mae', 
        random_state=42,
        verbose=-1
    ))
])

evaluate_model(
    "XGB",
    xgb_pipeline,
    X_train, y_train,
    X_test, y_test
)

evaluate_model(
    "CatBoost",
    cb_pipeline,
    X_train, y_train,
    X_test, y_test
)

evaluate_model(
    "LGB",
    lgb_pipeline,
    X_train, y_train,
    X_test, y_test
)

C:\Users\vensv\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [9]:
show_results()

,Model,Test R2,Test RMSE,Test MAE,Test MAPE,CV R2,CV RMSE,CV MAE
0,XGB,0.7696,944.8901,449.5513,0.5153,0.7291,1055.7057,472.1749
1,CatBoost,0.7547,974.7986,473.3637,0.5372,0.7267,1060.1630,484.8624
2,LGB,0.7692,945.6863,451.0279,0.5223,0.7302,1053.1835,468.0297


### Модель плохо сравляется с оценкой стоимости дорогих товаров.

In [10]:
y_pred = cb_pipeline.predict(X_test)
dif_column = abs(y_test-y_pred)
dif_df = pd.DataFrame({"test":y_test, "pred":y_pred, "dif":dif_column})
dif_df.sort_values("dif", ascending=False).head(30)

,test,pred,dif
10580,20142.45,8365.725773,11776.724227
8503,18271.48,7314.061747,10957.418253
7727,18592.74,7893.068259,10699.671741
3710,14500.00,4705.080413,9794.919587
5920,16576.56,7586.355184,8990.204816
2759,14500.00,5884.376272,8615.623728
9145,14249.40,5870.154902,8379.245098
2508,8500.00,344.046736,8155.953264
6898,11900.00,3937.379684,7962.620316
8431,16075.20,8137.588063,7937.611937


In [11]:
df_middle = df[df["price_byn"] < 7000]

y = df_middle["price_byn"]
X = df_middle.drop(["price_byn"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

In [13]:
show_results()

,Model,Test R2,Test RMSE,Test MAE,Test MAPE,CV R2,CV RMSE,CV MAE
0,XGB,0.7696,944.8901,449.5513,0.5153,0.7291,1055.7057,472.1749
1,CatBoost,0.7547,974.7986,473.3637,0.5372,0.7267,1060.1630,484.8624
2,LGB,0.7692,945.6863,451.0279,0.5223,0.7302,1053.1835,468.0297
3,XGB,0.8026,637.7488,366.7609,0.4849,0.8121,628.3948,369.5338
4,CatBoost,0.7896,658.4272,377.8188,0.4622,0.7997,648.7122,385.3493
5,LGB,0.8060,632.2741,365.9721,0.4858,0.8126,627.4798,370.3528


In [18]:
catboost_param_dist = {
    'regressor__iterations':           randint(400, 1200),
    'regressor__learning_rate':        uniform(0.01, 0.09),
    'regressor__depth':                randint(4, 10),
    'regressor__l2_leaf_reg':          uniform(1, 9),
    'regressor__subsample':            uniform(0.6, 0.4),
    'regressor__colsample_bylevel':    uniform(0.6, 0.4),
    'regressor__min_data_in_leaf':     randint(1, 50),
}

xgb_param_dist = {
    'regressor__n_estimators':         randint(400, 1200),
    'regressor__learning_rate':        uniform(0.01, 0.09),
    'regressor__max_depth':            randint(3, 10),
    'regressor__subsample':            uniform(0.6, 0.4),
    'regressor__colsample_bytree':     uniform(0.6, 0.4),
    'regressor__reg_alpha':            uniform(0, 5),
    'regressor__reg_lambda':           uniform(1, 9),
    'regressor__min_child_weight':     randint(1, 20),
}

lgb_param_dist = {
    'regressor__n_estimators':         randint(400, 1200),
    'regressor__learning_rate':        uniform(0.01, 0.09),
    'regressor__max_depth':            randint(3, 10),
    'regressor__num_leaves':           randint(20, 150),
    'regressor__subsample':            uniform(0.6, 0.4),
    'regressor__colsample_bytree':     uniform(0.6, 0.4),
    'regressor__reg_alpha':            uniform(0, 5),
    'regressor__reg_lambda':           uniform(1, 9),
    'regressor__min_child_samples':    randint(5, 50),
}

cat_search = RandomizedSearchCV(
    clone(cb_pipeline), catboost_param_dist,
    n_iter=10, scoring='neg_mean_absolute_error',
    cv=5, n_jobs=-1, random_state=42
)
cat_search.fit(X_train, y_train)

xgb_search = RandomizedSearchCV(
    clone(xgb_pipeline), xgb_param_dist,
    n_iter=10, scoring='neg_mean_absolute_error',
    cv=5, n_jobs=-1, random_state=42
)
xgb_search.fit(X_train, y_train)

lgb_search = RandomizedSearchCV(
    clone(lgb_pipeline), lgb_param_dist,
    n_iter=10, scoring='neg_mean_absolute_error',
    cv=5, n_jobs=-1, random_state=42
)
lgb_search.fit(X_train, y_train)

evaluate_model("CatBoost_tuned", cat_search.best_estimator_, X_train, y_train, X_test, y_test)
evaluate_model("XGB_tuned",      xgb_search.best_estimator_, X_train, y_train, X_test, y_test)
evaluate_model("LGB_tuned",      lgb_search.best_estimator_, X_train, y_train, X_test, y_test)

In [19]:
show_results()

,Model,Test R2,Test RMSE,Test MAE,Test MAPE,CV R2,CV RMSE,CV MAE
0,XGB,0.7696,944.8901,449.5513,0.5153,0.7291,1055.7057,472.1749
1,CatBoost,0.7547,974.7986,473.3637,0.5372,0.7267,1060.1630,484.8624
2,LGB,0.7692,945.6863,451.0279,0.5223,0.7302,1053.1835,468.0297
3,XGB,0.8026,637.7488,366.7609,0.4849,0.8121,628.3948,369.5338
4,CatBoost,0.7896,658.4272,377.8188,0.4622,0.7997,648.7122,385.3493
5,LGB,0.8060,632.2741,365.9721,0.4858,0.8126,627.4798,370.3528
6,CatBoost_tuned,0.7942,651.1117,374.7240,0.4679,0.8042,641.3890,381.7876
7,XGB_tuned,0.8179,612.4242,355.0936,0.4752,0.8252,606.0088,357.4174
8,LGB_tuned,0.8150,617.4458,357.4211,0.4725,0.8203,614.3739,363.6679


In [20]:
print("catboost\n", cat_search.best_params_)
print("xgb\n", xgb_search.best_params_)
print("lgbm\n", lgb_search.best_params_)

catboost
 {'regressor__colsample_bylevel': np.float64(0.9895022075365837), 'regressor__depth': 9, 'regressor__iterations': 1086, 'regressor__l2_leaf_reg': np.float64(6.565474083997786), 'regressor__learning_rate': np.float64(0.044421579214044646), 'regressor__min_data_in_leaf': 3, 'regressor__subsample': np.float64(0.9439761626945282)}
xgb
 {'regressor__colsample_bytree': np.float64(0.6733618039413735), 'regressor__learning_rate': np.float64(0.037381801866358394), 'regressor__max_depth': 8, 'regressor__min_child_weight': 12, 'regressor__n_estimators': 960, 'regressor__reg_alpha': np.float64(2.6238733012919457), 'regressor__reg_lambda': np.float64(4.598748745437299), 'regressor__subsample': np.float64(0.6186662652854461)}
lgbm
 {'regressor__colsample_bytree': np.float64(0.8080272084711243), 'regressor__learning_rate': np.float64(0.05920392514089517), 'regressor__max_depth': 8, 'regressor__min_child_samples': 33, 'regressor__n_estimators': 1102, 'regressor__num_leaves': 63, 'regressor__r

### Так как мы работаем с логнормальным распределнием попробуем логарифмирофать значение y_train.

In [24]:
y_train

535       450.00
6013     1490.00
10318     720.00
1500      500.00
7080     2367.98
          ...   
6195     1790.00
5594      269.00
5809     1450.00
898       900.00
7857     1868.98
Name: price_byn, Length: 7197, dtype: float64

In [35]:
log_y_train = np.log(y_train)

# log_cat_search = RandomizedSearchCV(
#     clone(cb_pipeline), catboost_param_dist,
#     n_iter=30, scoring='neg_mean_absolute_error',
#     cv=5, n_jobs=-1, random_state=42
# )
# log_cat_search.fit(X_train, log_y_train)

# log_xgb_search = RandomizedSearchCV(
#     clone(xgb_pipeline), xgb_param_dist,
#     n_iter=30, scoring='neg_mean_absolute_error',
#     cv=5, n_jobs=-1, random_state=42
# )
# log_xgb_search.fit(X_train, log_y_train)

log_lgb_search = RandomizedSearchCV(
    clone(lgb_pipeline), lgb_param_dist,
    n_iter=60, scoring='neg_mean_absolute_error',
    cv=5, n_jobs=-1, random_state=42
)
log_lgb_search.fit(X_train, log_y_train)

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('extractor',
                                              FeatureExtractor(GBM_XGB=True,
                                                               access_time=Timestamp('2026-04-26 13:10:00+0300', tz='Europe/Moscow'))),
                                             ('regressor',
                                              LGBMRegressor(learning_rate=0.03,
                                                            max_depth=7,
                                                            n_estimators=1000,
                                                            objective='mae',
                                                            random_state=42,
                                                            verbose=-1))]),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'regressor__colsample_b...
                                        'regressor__reg_alpha': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001F7A6961A20>,
                                        'regressor__reg_lambda': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001F7A6961EF0>,
                                        'regressor__subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x000001F7B826C9F0>},
                   random_state=42, scoring='neg_mean_absolute_error')

In [36]:
# log_model(log_cat_search, X_train, log_y_train, X_test, y_test, run_name="log_cat_search21.05.2026", log=True)
# log_model(log_xgb_search, X_train, log_y_train, X_test, y_test, run_name="log_xgb_search21.05.2026", log=True)
log_model(log_lgb_search, X_train, log_y_train, X_test, y_test, run_name="log_lgb_search21.05.2026", log=True)

2026/05/21 23:43:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/21 23:43:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [37]:
# evaluate_model("log_CatBoost_tuned", log_cat_search.best_estimator_, X_train, y_train, X_test, y_test)
# evaluate_model("log_XGB_tuned",      log_xgb_search.best_estimator_, X_train, y_train, X_test, y_test)
evaluate_model("log_LGB_tuned",      log_lgb_search.best_estimator_, X_train, y_train, X_test, y_test)

In [38]:
show_results()

,Model,Test R2,Test RMSE,Test MAE,Test MAPE,CV R2,CV RMSE,CV MAE
0,XGB,0.7696,944.8901,449.5513,0.5153,0.7291,1055.7057,472.1749
1,CatBoost,0.7547,974.7986,473.3637,0.5372,0.7267,1060.1630,484.8624
2,LGB,0.7692,945.6863,451.0279,0.5223,0.7302,1053.1835,468.0297
3,XGB,0.8026,637.7488,366.7609,0.4849,0.8121,628.3948,369.5338
4,CatBoost,0.7896,658.4272,377.8188,0.4622,0.7997,648.7122,385.3493
5,LGB,0.8060,632.2741,365.9721,0.4858,0.8126,627.4798,370.3528
6,CatBoost_tuned,0.7942,651.1117,374.7240,0.4679,0.8042,641.3890,381.7876
7,XGB_tuned,0.8179,612.4242,355.0936,0.4752,0.8252,606.0088,357.4174
8,LGB_tuned,0.8150,617.4458,357.4211,0.4725,0.8203,614.3739,363.6679
9,log_CatBoost_tuned,0.7942,651.1117,374.7240,0.4679,0.8042,641.3890,381.7876
